# Phase 3: Collaborative Filtering & Matrix Factorization

In this notebook, we explore **Collaborative Filtering** models: User-User CF, Item-Item CF, and SVD Matrix Factorization, and compare them against our Phase 1 Popularity and Phase 2 Content-Based models.

In [ ]:
import pandas as pd
import numpy as np
import sys
import os

sys.path.append(os.path.abspath('../src'))
from collaborative import build_user_item_matrix, UserBasedRecommender, ItemBasedRecommender, MatrixFactorizationRecommender
from content_based import ContentBasedRecommender
from preprocessing import PopularityRecommender

# Load datasets
df_users = pd.read_csv('../data/raw/users.csv')
df_courses = pd.read_csv('../data/raw/courses.csv')
df_interactions = pd.read_csv('../data/raw/interactions.csv')

matrix, stats = build_user_item_matrix(df_interactions)
print(f"User-Item Matrix Shape: {matrix.shape} | Sparsity: {stats['sparsity_pct']:.2f}%")

## 1. User-User Collaborative Filtering Recommendations
Recommendations for User `U001` (Frontend Persona) based on similar users.

In [ ]:
ub_rec = UserBasedRecommender(k_neighbors=10).fit(df_interactions, df_courses)
ub_recs_u001 = ub_rec.recommend('U001', top_n=5)
ub_recs_u001[['course_id', 'title', 'category', 'predicted_score', 'recommendation_type']]

## 2. Item-Item Collaborative Similar Courses vs Content-Based Similar Courses
Comparing course similarity results for Course `C06` (Python Programming).

In [ ]:
ib_rec = ItemBasedRecommender().fit(df_interactions, df_courses)
cb_rec = ContentBasedRecommender().fit(df_courses, df_interactions)

print("=== Collaborative Item Similarity for C06 ===")
display(ib_rec.recommend_collaborative_similar_courses('C06', top_n=3)[['course_id', 'title', 'category', 'similarity_score']])

print("=== Content-Based Item Similarity for C06 ===")
display(cb_rec.recommend_similar_courses('C06', top_n=3)[['course_id', 'title', 'category', 'similarity_score']])

## 3. Matrix Factorization via Truncated SVD

In [ ]:
svd_rec = MatrixFactorizationRecommender(n_components=5).fit(df_interactions, df_courses)
svd_recs_u001 = svd_rec.recommend('U001', top_n=5)
svd_recs_u001[['course_id', 'title', 'category', 'predicted_score', 'recommendation_type']]

## 4. Comprehensive Offline Evaluation across All Recommender Models (N=50 Users)

In [ ]:
# Perform Leave-Last-Course-Out Split
train_rows = []
test_lookup = {}

for user_id, group in df_interactions.groupby('user_id'):
    unique_courses = group['course_id'].unique()
    if len(unique_courses) >= 2:
        held_out_course = unique_courses[-1]
        test_lookup[user_id] = held_out_course
        train_rows.append(group[group['course_id'] != held_out_course])
    else:
        train_rows.append(group)

df_train = pd.concat(train_rows, ignore_index=True)

# Fit All Models on Train Set
m_pop = PopularityRecommender()
df_imp = df_train[df_train['interaction_type'] != 'RATE'].copy()
df_imp['weight'] = df_imp['interaction_type'].map({'VIEW': 1.0, 'BOOKMARK': 2.0, 'ENROLL': 3.0, 'COMPLETE': 4.0})
summary = df_imp.groupby(['user_id', 'course_id'])['weight'].sum().reset_index()
m_pop.fit(summary, df_courses)

m_cb = ContentBasedRecommender().fit(df_courses, df_train)
m_ub = UserBasedRecommender(k_neighbors=10).fit(df_train, df_courses)
m_ib = ItemBasedRecommender(top_k_similar=10).fit(df_train, df_courses)
m_svd = MatrixFactorizationRecommender(n_components=5).fit(df_train, df_courses)

models = {
    'Popularity Baseline': m_pop,
    'Content-Based': m_cb,
    'User-User CF': m_ub,
    'Item-Item CF': m_ib,
    'SVD Matrix Factorization': m_svd
}

results = []
n_eval = len(test_lookup)

for name, model in models.items():
    hits = 0
    for uid, target_c in test_lookup.items():
        recs = model.recommend(uid, top_n=5)['course_id'].tolist()
        if target_c in recs:
            hits += 1
    precision = hits / (n_eval * 5)
    recall = hits / n_eval
    results.append({'Model': name, 'Precision@5': precision, 'Recall@5': recall, 'Total Hits': hits})

df_res = pd.DataFrame(results)
print(f"=== OFFLINE EVALUATION RESULTS (N={n_eval} Users) ===")
display(df_res)